<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Course</h1>
<h1>Lecture</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
from pprint import pprint
import json

import pandas as pd
import numpy as np

import openai
from openai import OpenAI

import termcolor
from termcolor import colored

import watermark

%load_ext watermark
%matplotlib inline

We start by printing out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.4
IPython version      : 8.12.3

Compiler    : Clang 14.0.6 
OS          : Darwin
Release     : 23.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: fbdcbe829c7e7175d5b838e71a1f15311b0102b2

pandas   : 2.2.3
termcolor: 2.5.0
watermark: 2.4.3
openai   : 1.54.2
numpy    : 1.26.4
json     : 2.0.9



# Basic Usage

The first step is generate API key on the OpenAI website and store it as the "OPENAI_API_KEY" variable in your local environment. Without it we won't be able to do anything. You can find your API key in your using settings: https://help.openai.com/en/articles/4936850-where-do-i-find-my-secret-api-key

Then we are ready to instantiate the client

In [3]:
client = OpenAI()

We start by getting a list of supported models.

In [4]:
model_list = json.loads(client.models.list().json())["data"]

/var/folders/lr/j1bs1q851k15cj5y777nxwph0000gn/T/ipykernel_99898/953273611.py:1: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.5/migration/
  model_list = json.loads(client.models.list().json())["data"]


In total we have 38 models

In [5]:
len(model_list)

38

Along with some information about each model...

In [6]:
model_list[:5]

[{'id': 'gpt-4-turbo-2024-04-09',
  'created': 1712601677,
  'object': 'model',
  'owned_by': 'system'},
 {'id': 'tts-1-1106',
  'created': 1699053241,
  'object': 'model',
  'owned_by': 'system'},
 {'id': 'dall-e-2',
  'created': 1698798177,
  'object': 'model',
  'owned_by': 'system'},
 {'id': 'whisper-1',
  'created': 1677532384,
  'object': 'model',
  'owned_by': 'openai-internal'},
 {'id': 'gpt-3.5-turbo-instruct',
  'created': 1692901427,
  'object': 'model',
  'owned_by': 'system'}]

But let's just get a list of model names

In [7]:
print("\n".join(sorted([model["id"] for model in model_list])))

babbage-002
chatgpt-4o-latest
dall-e-2
dall-e-3
davinci-002
gpt-3.5-turbo
gpt-3.5-turbo-0125
gpt-3.5-turbo-0301
gpt-3.5-turbo-0613
gpt-3.5-turbo-1106
gpt-3.5-turbo-16k
gpt-3.5-turbo-16k-0613
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
gpt-4
gpt-4-0125-preview
gpt-4-0613
gpt-4-1106-preview
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4-turbo-preview
gpt-4o
gpt-4o-2024-05-13
gpt-4o-2024-08-06
gpt-4o-audio-preview
gpt-4o-audio-preview-2024-10-01
gpt-4o-mini
gpt-4o-mini-2024-07-18
gpt-4o-realtime-preview
gpt-4o-realtime-preview-2024-10-01
text-embedding-3-large
text-embedding-3-small
text-embedding-ada-002
tts-1
tts-1-1106
tts-1-hd
tts-1-hd-1106
whisper-1


## Basic Prompt

The recommended model for exploration is `gpt-3.5-turbo` (as it's the cheapest), so we'll stick with it for now. The basic setup is relatively straightforward:

In [8]:
%%time
response = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=[
        {
            "role": "user", 
            "content": "What was Superman's weakness?"
        },
    ]
)

CPU times: user 13.4 ms, sys: 3.08 ms, total: 16.4 ms
Wall time: 989 ms


Which produces a response object

In [9]:
type(response)

openai.types.chat.chat_completion.ChatCompletion

In [29]:
response.usage

CompletionUsage(completion_tokens=16, prompt_tokens=12, total_tokens=28, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

Which we can treat as a named tuple

The model answer can be found in the "message" dictionary inside the "choices" list

In [10]:
response.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Superman's weakness was the mineral kryptonite, a radioactive element from his home planet of Krypton. Exposure to kryptonite weakened Superman and could ultimately kill him if he was exposed to it for too long.", refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))

In [11]:
response.choices[0].message.content

"Superman's weakness was the mineral kryptonite, a radioactive element from his home planet of Krypton. Exposure to kryptonite weakened Superman and could ultimately kill him if he was exposed to it for too long."

To request multiple answers, we must include the `n` parameter with the number of answers we want

In [12]:
%%time
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "What are the different kinds of Kryptonite?"},
    ],
    n=3
)

CPU times: user 6.85 ms, sys: 2.34 ms, total: 9.19 ms
Wall time: 4.39 s


And we can access each of the answers individually int he choices list

In [13]:
for output in response.choices:
    print("==========")
    print(output.message.role.title()) 
    print("==========")
    print(output.message.content)
    print("==========\n")

Assistant
There are several different kinds of Kryptonite in the DC Comics universe, each with varying effects on Superman and other Kryptonians. Some of the most well-known types include:

1. Green Kryptonite - This is the most common form of Kryptonite and is deadly to Kryptonians. Exposure to green Kryptonite weakens Superman and can ultimately kill him if he is exposed to it for an extended period of time.

2. Red Kryptonite - Red Kryptonite has unpredictable effects on Kryptonians, causing temporary changes in their powers, personalities, or physical appearances. These effects are different each time a Kryptonian is exposed to red Kryptonite.

3. Gold Kryptonite - Gold Kryptonite permanently removes a Kryptonian's powers, leaving them permanently vulnerable to harm. It is often considered one of the most dangerous forms of Kryptonite.

4. Blue Kryptonite - Blue Kryptonite is specifically harmful to Bizarro, a flawed clone of Superman. It has little effect on Superman himself.

5. 

In [14]:
response.usage

CompletionUsage(completion_tokens=912, prompt_tokens=17, total_tokens=929, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

# Temperature

In [15]:
%%time
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "Tell me a dad joke"},
    ],
    temperature=1.8
)

CPU times: user 6.07 ms, sys: 1.81 ms, total: 7.88 ms
Wall time: 576 ms


In [16]:
print(response.choices[0].message.content)

Why did the scarecrow win an award? Because he was outstanding in his field!


In [17]:
%%time
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "Tell me a dad joke"},
    ],
    temperature=0
)

CPU times: user 7.81 ms, sys: 1.71 ms, total: 9.52 ms
Wall time: 601 ms


In [18]:
print(response.choices[0].message.content)

Why couldn't the bicycle stand up by itself?

Because it was two tired!


# Function Calls

In [19]:
def chat(messages, functions):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        # Define the functions the model is allowed to use
        functions=functions
    )
    
    return response

In [20]:
def pretty_print_conversation(messages):
    role_to_color = {
        "system": "red",
        "user": "green",
        "assistant": "blue",
        "function": "magenta",
    }
    
    for message in messages:
#         print(message)
        if message["role"] == "system":
            print(colored(f"system: {message['content']}\n", role_to_color[message['role']]))
        elif message["role"] == "user":
            print(colored(f"user: {message['content']}\n", role_to_color[message['role']]))
        elif message["role"] == "assistant" and message['function_call']:
            print(colored(f"assistant: {message['function_call']}\n", role_to_color[message['role']]))
        elif message["role"] == "assistant" and not message['function_call']:
            print(colored(f"assistant: {message['content']}\n", role_to_color[message['role']]))
        elif message["role"] == "function":
            print(colored(f"function ({message.name}): {message.content}\n", role_to_color[message.role]))


Let's create some function specifications to interface with a hypothetical weather API. We'll pass these function specification to the Chat Completions API in order to generate function arguments that adhere to the specification.

In [21]:
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "format": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "The temperature unit to use. Infer this from the users location.",
                },
            },
            "required": ["location", "format"],
        },
    },
]

If we prompt the model about the current weather, it will respond with some clarifying questions.

In [22]:
messages = []

messages.append(
    {"role": "system", 
     "content": "Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous."
    })

messages.append(
    {"role": "user", 
     "content": "What's the weather like today"
    })

In [23]:
chat_response = chat(messages, functions=functions)

assistant_message = chat_response.choices[0].message

messages.append({
 "role":  assistant_message.role,
 "content":  assistant_message.content,
 "function_call":  assistant_message.function_call,
})

In [24]:
pretty_print_conversation(messages)

system: Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.

user: What's the weather like today

assistant: Sure, I can help with that. Could you please provide me with your current location so I can get the weather information for you?



Once we provide the missing information, it will generate the appropriate function arguments for us.

In [25]:
messages.append(
    {"role": "user", 
     "content": "I'm in New York, NY."
    })

In [26]:
chat_response = chat(messages, functions=functions)

assistant_message = chat_response.choices[0].message

messages.append({
 "role":  assistant_message.role,
 "content":  assistant_message.content,
 "function_call":  assistant_message.function_call,
})

In [27]:
pretty_print_conversation(messages)

system: Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.

user: What's the weather like today

assistant: Sure, I can help with that. Could you please provide me with your current location so I can get the weather information for you?

user: I'm in New York, NY.

assistant: FunctionCall(arguments='{"location":"New York, NY","format":"celsius"}', name='get_current_weather')



<div style="width: 100%; overflow: hidden;">
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</div>